# Εργασία Ανάκτησης Πληροφορίας

Οι βιβλιοθήκες που χρειάζονται είναι στο README.

Το βήμα 1 κατεβάζει 50 νέα άρθρα από τη Wikipedia και γράφει από πάνω το data.json. Τα σχετικά άρθρα στο βήμα 5 είναι για τα άρθρα που υπάρχουν ήδη στο data.json, οπότε για τα ίδια αποτελέσματα με το README ξεκίνα από το βήμα 2.

## Βήμα 1. Συλλογή δεδομένων

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

url = "https://en.wikipedia.org/wiki/Information_retrieval"

max_articles = 50

articles = []
visited_url = set()
url_scrape = {url}

while url_scrape and len(articles) < max_articles:
    current_url = url_scrape.pop()
    
    if current_url in visited_url:
        continue

    try:
        response = requests.get(current_url, headers={"User-Agent": "IR-LabProject/1.0 (student project)"})
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')

        article_title = soup.find('h1', {'id': 'firstHeading'}).text.strip()
        paragraphs = soup.find_all('p')
        content = " ".join([para.text for para in paragraphs if para.text])
        
        articles.append({
            "title": article_title,
            "url": current_url,
            "content": content})

        visited_url.add(current_url)

        for link in soup.find_all('a', href=True):
            links = link['href'].replace("https://en.wikipedia.org", "")
            if links.startswith('/wiki/') and not links.startswith('/wiki/Special:') and ':' not in links:
                full_url = f"https://en.wikipedia.org{links}"
                if full_url not in visited_url:
                    url_scrape.add(full_url)
    
    except requests.RequestException:
        print("Request failed")

with open("data.json", "w") as file:
    json.dump(articles, file, indent=4)

## Βήμα 2. Προεπεξεργασία κειμένου

In [ ]:
import json
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

try:
    with open("data.json", "r") as file:
        data = json.load(file)
except FileNotFoundError:
    print("File not found")
    exit(0)
    
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

processed_data = []

for article in data:
    content = article.get("content", "").strip()
    if not content:
        continue
    
    tokenized_words = word_tokenize(content)
    tokenized_words = [word.lower() for word in tokenized_words]
    tokenized_words = [word for word in tokenized_words if word.isalpha()]
    tokenized_words = [word for word in tokenized_words if word not in stop_words]
    tokenized_words = [lemmatizer.lemmatize(word) for word in tokenized_words]
    
    processed_data.append(" ".join(tokenized_words))

with open("processed_data.json", "w") as file:
    json.dump(processed_data, file, indent=4)

## Βήμα 3. Ευρετήριο

In [ ]:
import json

try:
    with open("processed_data.json", "r") as file:
        data = json.load(file)
except FileNotFoundError:
    print("File not found")
    exit(0)

inverted_index = {}

for doc_id, article in enumerate(data, start=1):

    words = article.split()
    for word in words:
        if word not in inverted_index:
            inverted_index[word] = {}
        if doc_id not in inverted_index[word]:
            inverted_index[word][doc_id] = 0
        inverted_index[word][doc_id] += 1

with open("inverted_index_data.json", "w") as file:
    json.dump(inverted_index, file, indent=4)

## Βήμα 4. Μηχανή αναζήτησης

In [ ]:
import json
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi


def load_data():
    try:
        with open("data.json", "r") as data_file:
            data = json.load(data_file)

        with open("inverted_index_data.json", "r") as inverted_index_file:
            inverted_index = json.load(inverted_index_file)

        with open("processed_data.json", "r") as processed_data_file:
            processed_data = json.load(processed_data_file)

        return data, processed_data, inverted_index

    except FileNotFoundError:
        print("Error loading files")

        return None, None, None


def preprocess_query(query):
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words("english"))
    tokens = word_tokenize(query.lower())
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens


def boolean_search(query, inverted_index, operator):
    all_ids = sorted({int(i) for index in inverted_index.values() for i in index.keys()})

    result_lists = [{int(doc_id) for doc_id in inverted_index.get(term, {}).keys()} for term in query]

    if operator == "AND":
        return list(set.intersection(*result_lists)) if result_lists else []

    elif operator == "OR":
        return sorted(set.union(*result_lists)) if result_lists else []

    elif operator == "NOT":
        result_union = set.union(*result_lists) if result_lists else set()
        return [i for i in all_ids if i not in result_union]


def tfidf_ranking(query, processed_data):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(processed_data)
    query_vector = vectorizer.transform([" ".join(query)])

    scores = np.dot(tfidf_matrix, query_vector.T).toarray().flatten()
    rank = np.argsort(scores)
    rank = np.flip(rank)

    result = []
    for i in rank:
        if scores[i] > 0:
            result.append((i, scores[i]))

    return result


def bm25_ranking(query, processed_data):
    tokenized_data = [data.split() for data in processed_data]

    bm25 = BM25Okapi(tokenized_data)
    scores = bm25.get_scores(query)

    rank = np.argsort(scores)
    rank = np.flip(rank)

    result = []
    for i in rank:
        if scores[i] > 0:
            result.append((i, scores[i]))

    return result

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from backend_search_engine import load_data, preprocess_query, boolean_search, tfidf_ranking, bm25_ranking

# τα json αρχεία διαβάζονται από τον φάκελο του notebook
data, processed_data, inverted_index = load_data()

def format_results(results, data):
    # tf-idf και bm25 δίνουν (θέση, score), οι θέσεις ξεκινούν από το 0
    if isinstance(results, list) and results and isinstance(results[0], tuple): 
        return "<br>\n".join(
            [
                f'<a href="{data[doc_id].get("url", "#")}" target="_blank">'
                f'{data[doc_id].get("title", "Untitled Document")}</a> '
                f"(Score: {score:.4f})"
                for doc_id, score in results
                if 0 <= doc_id < len(data)
            ]
        )
    elif isinstance(results, (list, set)) and results and isinstance(next(iter(results)), int):  # αποτελέσματα boolean
        # τα ids του boolean ξεκινούν από το 1, γι' αυτό doc_id - 1
        return "<br>\n".join(
            [
                f'<a href="{data[doc_id - 1].get("url", "#")}" target="_blank">'
                f'{data[doc_id - 1].get("title", "Untitled Document")}</a>'
                for doc_id in sorted(results)
                if 0 <= (doc_id - 1) < len(data)
            ]
        )
    return "No matching documents found."


def search_interface(query, operator, algorithm):
    if not data or not processed_data or not inverted_index:
        return "Data files not loaded correctly."

    query_terms = preprocess_query(query)

    # ο operator (AND, OR, NOT) χρησιμοποιείται μόνο στο Boolean
    if algorithm == "Boolean":
        results = boolean_search(query_terms, inverted_index, operator)
        return format_results(results, data) if results else "No matching documents found."
        
    elif algorithm == "TF-IDF":
        results = tfidf_ranking(query_terms, processed_data)
        return format_results(results, data) if results else "No matching documents found."
        
    elif algorithm == "BM25":
        results = bm25_ranking(query_terms, processed_data)
        return format_results(results, data) if results else "No matching documents found."


# τα πεδία του interface και το output για τα αποτελέσματα
query_input = widgets.Text(description="Query:", placeholder="Enter your query")
operator_input = widgets.Dropdown(options=["AND", "OR", "NOT"], description="Operator:")
algorithm_input = widgets.Dropdown(options=["Boolean", "TF-IDF", "BM25"], description="Algorithm:")
output = widgets.Output()

# όταν πατηθεί το Search, σβήνει τα προηγούμενα αποτελέσματα και δείχνει τα νέα
def on_search_click(_):
    with output:
        output.clear_output()
        results = search_interface(query_input.value, operator_input.value, algorithm_input.value)
        display(HTML(results))

search_button = widgets.Button(description="Search")
search_button.on_click(on_search_click)

display(query_input, operator_input, algorithm_input, search_button, output)

## Βήμα 5. Αξιολόγηση συστήματος

In [ ]:
import json

from sklearn.metrics import precision_score, recall_score, f1_score

from backend_search_engine import preprocess_query, boolean_search, tfidf_ranking, bm25_ranking


def load_data():
    try:
        with open("inverted_index_data.json", "r") as inverted_index_file:
            inverted_index = json.load(inverted_index_file)

        with open("processed_data.json", "r") as processed_data_file:
            processed_data = json.load(processed_data_file)

        return processed_data, inverted_index

    except FileNotFoundError:
        print("Error loading files")

        return None, None


def calculate_evaluations(retrieved_docs, relevant_docs, total_docs):
    # τα έγγραφα αριθμούνται από το 1 έως total_docs
    y_true = [1 if i in relevant_docs else 0 for i in range(1, total_docs + 1)]
    y_pred = [1 if i in retrieved_docs else 0 for i in range(1, total_docs + 1)]

    precision = precision_score(y_true, y_pred, zero_division=0)

    recall = recall_score(y_true, y_pred, zero_division=0)

    f1 = f1_score(y_true, y_pred, zero_division=0)

    return precision, recall, f1


# πόσα από τα πρώτα k αποτελέσματα είναι σχετικά
def calculate_precision_at_k(ranked_docs, relevant_docs, k):
    hits = [doc for doc in ranked_docs[:k] if doc in relevant_docs]

    return len(hits) / k


# κάθε φορά που βρίσκουμε σχετικό κρατάμε το precision μέχρι εκείνη τη θέση,
# όσα σχετικά δεν βρέθηκαν μετράνε 0
def calculate_average_precision(ranked_docs, relevant_docs):
    found = 0
    total = 0

    for i in range(len(ranked_docs)):
        if ranked_docs[i] in relevant_docs:
            found += 1
            total += found / (i + 1)

    return total / len(relevant_docs)


def main():
    processed_data, inverted_index = load_data()

    if not processed_data or not inverted_index:
        print("Error loading data. Exiting.")
        return

    queries = [
        "information retrieval",
        "machine learning",
        "women's rights",
        "military coup",
        "television broadcasting",
        "environmental sustainability",
    ]

    # σχετικά είναι τα άρθρα που μιλάνε για το θέμα, όχι όσα έχουν απλώς τις λέξεις
    relevant_docs = {
        "information retrieval": [1, 3],
        "machine learning": [2],
        "women's rights": [9, 15, 17, 18, 28, 32],
        "military coup": [36, 38, 49],
        "television broadcasting": [10, 19, 24, 31],
        "environmental sustainability": [29, 30, 46],
    }

    total_docs = len(processed_data)

    tfidf_average_precisions = []
    bm25_average_precisions = []

    for query in queries:
        query_terms = preprocess_query(query)

        # το NOT δεν το αξιολογούμε, δίνει τα άρθρα που δεν έχουν τους όρους
        and_result = boolean_search(query_terms, inverted_index, "AND")
        or_result = boolean_search(query_terms, inverted_index, "OR")
        # το tf-idf και το bm25 δίνουν θέσεις από το 0, τις κάνουμε ids από το 1
        tfidf_result = [doc_id + 1 for doc_id, _ in tfidf_ranking(query_terms, processed_data)]
        bm25_result = [doc_id + 1 for doc_id, _ in bm25_ranking(query_terms, processed_data)]

        print(f"\nEvaluation for: {query}")

        # το boolean δίνει λίστα χωρίς σειρά
        for method, result in [("BOOLEAN AND", and_result), ("BOOLEAN OR", or_result)]:
            precision, recall, f1 = calculate_evaluations(result, relevant_docs[query], total_docs)

            print(f"\n{method}:")
            print(f"Precision: {precision:.4f}")
            print(f"Recall: {recall:.4f}")
            print(f"F1-Score: {f1:.4f}")

        # στο tf-idf και στο bm25 μετράει και η σειρά
        for method, result in [("TF-IDF", tfidf_result), ("BM25", bm25_result)]:
            precision_at_5 = calculate_precision_at_k(result, relevant_docs[query], 5)
            average_precision = calculate_average_precision(result, relevant_docs[query])

            if method == "TF-IDF":
                tfidf_average_precisions.append(average_precision)
            else:
                bm25_average_precisions.append(average_precision)

            print(f"\n{method}:")
            print(f"Precision at 5: {precision_at_5:.4f}")
            print(f"Average Precision: {average_precision:.4f}")

    # το mean average precision είναι ο μέσος όρος των average precision όλων των queries
    tfidf_mean_average_precision = sum(tfidf_average_precisions) / len(tfidf_average_precisions)
    bm25_mean_average_precision = sum(bm25_average_precisions) / len(bm25_average_precisions)

    print("\nMean Average Precision:")
    print(f"TF-IDF: {tfidf_mean_average_precision:.4f}")
    print(f"BM25: {bm25_mean_average_precision:.4f}")


if __name__ == "__main__":
    main()